# `tt_fft` — PyTorch-style FFT on Tenstorrent Wormhole

Drop-in replacement for `torch.fft.fft` / `np.fft.fft`, backed by the universal FFT pipeline (`fft_universal` for fp32 and `fft_universal_bf16` for true bf16 FPU compute).

**Both the input AND the FFT run on Tenstorrent**: `tt_fft.rand(N)` / `tt_fft.randn(N)` allocate the signal via `ttnn.rand(...)` on the device, then `tt_fft.fft(x)` runs the universal FFT kernels. (If `ttnn` isn't available in the env, the input generators fall back to numpy automatically.)

**Accepts ANY length N ≥ 2** — the device side picks the right algorithm:

| N pattern        | Algorithm                                 |
|------------------|-------------------------------------------|
| `N ≤ 32`         | Packed direct-DFT (single tile, FPU matmul) |
| pow2             | Stockham (fp32) / two-level Cooley-Tukey (bf16) |
| prime > 32       | Bluestein chirp-Z transform               |
| composite non-pow2 | Mixed-radix Cooley-Tukey                |

**Two precisions:** `precision='fp32'` (default) or TRUE `precision='bf16'` (FPU bf16 multiplier + fp32 accumulator).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import tt_fft

print('tt_fft loaded — binaries:')
print('  fp32 :', tt_fft._BIN_FP32)
print('  bf16 :', tt_fft._BIN_BF16)
print('  ttnn :', 'available' if tt_fft._HAVE_TTNN else 'not installed (input generators will use numpy)')

## 1. The PyTorch-style API — input AND FFT both on Tenstorrent

Same call signature as `torch.fft.fft`. Both lines below use Tenstorrent: `tt_fft.randn` allocates via `ttnn.rand`, `tt_fft.fft` runs `fft_universal`.

In [ ]:
x = tt_fft.randn(1024, complex=True, seed=42)   # input on Tenstorrent (ttnn.rand)
X = tt_fft.fft(x)                               # FFT  on Tenstorrent (fft_universal)
y = tt_fft.ifft(X)                              # IFFT on Tenstorrent

X_torch = torch.fft.fft(torch.from_numpy(x)).numpy()    # CPU reference for the audience
print(f'max abs diff (Wormhole vs PyTorch) : {np.max(np.abs(X - X_torch)):.3e}')
print(f'round-trip error  ifft(fft(x))     : {np.max(np.abs(y - x)):.3e}')

## 2. Works on ANY N — dispatch tree picks the algorithm

In [ ]:
for N in [16, 60, 97, 1000, 1024, 4096, 65536]:
    print(f'  N = {N:>6d}   ->   {tt_fft.device_path(N)}')

In [ ]:
# Prime (Bluestein), composite non-pow2 (mixed-radix), pow2 (Stockham).
for N in [97, 60, 4096]:
    x = tt_fft.randn(N, complex=True, seed=N)        # input on Tenstorrent
    X_ref = np.fft.fft(x)
    X_tt  = tt_fft.fft(x)                            # FFT on Tenstorrent
    rel   = np.max(np.abs(X_tt - X_ref)) / np.max(np.abs(X_ref))
    print(f'  N={N:>5d}  path={tt_fft.device_path(N):<48s}  rel err = {rel:.2e}')

## 3. Visual: pure tone -> single spike

Input is `tt_fft.tone(N, k)` (a deterministic complex tone), FFT runs on Tenstorrent, expected magnitude at bin `k` is `N`.

In [ ]:
N, k = 1024, 17
x = tt_fft.tone(N, k=k)
X = tt_fft.fft(x)

plt.figure(figsize=(11, 4))
plt.stem(np.abs(X), basefmt=' ', markerfmt='.', linefmt='-')
plt.axvline(k, color='red', ls='--', alpha=0.4, label=f'input bin k={k}')
plt.title(f'tt_fft.fft of pure tone at bin {k}  (N={N})')
plt.xlabel('bin'); plt.ylabel('|X[k]|'); plt.legend()
plt.tight_layout(); plt.show()
print(f'|X[{k}]| = {np.abs(X[k]):.2f}   (expected = N = {N})')

## 4. Real signal: chord of three sines

In [ ]:
N = 4096
freqs, amps = (50, 120, 240), (1.0, 0.6, 0.3)
x = tt_fft.chord(N, freqs=freqs, amps=amps, noise=0.02, seed=7)
X = tt_fft.fft(x)
mag = np.abs(X[:N//2])

fig, axs = plt.subplots(2, 1, figsize=(11, 6))
axs[0].plot(np.arange(512) / N, x[:512])
axs[0].set_title('input signal (first 512 samples)')
axs[0].set_xlabel('time (s)'); axs[0].set_ylabel('amplitude')
axs[1].plot(mag)
for f in freqs: axs[1].axvline(f, color='red', ls='--', alpha=0.4)
axs[1].set_title('|tt_fft.fft(x)|  (one-sided)  -- red lines = expected peaks')
axs[1].set_xlabel('bin'); axs[1].set_ylabel('magnitude')
plt.tight_layout(); plt.show()

## 5. fp32 vs TRUE bf16 — same API, just change `precision`

In [ ]:
N = 1024
x = tt_fft.randn(N, complex=True, seed=0)

X_ref  = np.fft.fft(x)
X_fp32 = tt_fft.fft(x, precision='fp32')
X_bf16 = tt_fft.fft(x, precision='bf16')

def snr(ref, got):
    diff = got.astype(np.complex128) - ref
    return 10 * np.log10(np.sum(np.abs(ref)**2) / max(float(np.sum(np.abs(diff)**2)), 1e-300))

print(f'  tt_fft.fft(x, precision="fp32")  SNR vs numpy : {snr(X_ref, X_fp32):6.2f} dB')
print(f'  tt_fft.fft(x, precision="bf16")  SNR vs numpy : {snr(X_ref, X_bf16):6.2f} dB')

## 6. Round-trip:  x -> FFT -> IFFT -> x

In [ ]:
N = 1000  # composite non-pow2 -> mixed-radix path
x = tt_fft.randn(N, complex=True, seed=1)

X = tt_fft.fft(x)
y = tt_fft.ifft(X)

rel = np.max(np.abs(y - x)) / np.max(np.abs(x))
print(f'N = {N}, dispatch = {tt_fft.device_path(N)}')
print(f'round-trip max rel error : {rel:.3e}')

plt.figure(figsize=(11, 3))
plt.plot(x.real[:128],   label='input.real',         lw=2,   alpha=0.8)
plt.plot(y.real[:128], '--', label='ifft(fft(x)).real', lw=1.2, alpha=0.9)
plt.legend(); plt.title('Round-trip first 128 samples'); plt.tight_layout(); plt.show()

## 7. End-to-end timing across N

In [ ]:
rows = []
for N in [32, 100, 512, 1024, 4096, 16384, 65536]:
    r = tt_fft.benchmark(N, iters=10, precision='fp32')
    rows.append(r)
    print(f'N={N:>6d}  warm avg {r["warm_avg_ms"]:7.2f} ms  '
          f'cold {r["cold_ms"]:7.1f} ms  SNR {r["snr_db"]:6.1f} dB  '
          f'[{r["dispatch"]}]')

Ns = [r['N'] for r in rows]
ms = [r['warm_avg_ms'] for r in rows]
plt.figure(figsize=(7, 4))
plt.loglog(Ns, ms, 'o-', lw=2)
plt.xlabel('N'); plt.ylabel('warm avg ms (end-to-end)')
plt.title('tt_fft.fft  end-to-end time vs N')
plt.grid(True, which='both', alpha=0.3); plt.tight_layout(); plt.show()

---
### Summary

* **Both input and FFT run on Tenstorrent**: `tt_fft.randn / tt_fft.tone / tt_fft.chord` use `ttnn` on the device, `tt_fft.fft / tt_fft.ifft` run the universal FFT pipeline.
* **Same shape as `torch.fft`**: `tt_fft.fft(x)`, `tt_fft.ifft(X)`, `tt_fft.rfft(x)`, `tt_fft.fft2(img)`.
* **Any N ≥ 2**: pow2, prime, composite — dispatcher picks the algorithm.
* **Two precisions**: `'fp32'` and TRUE `'bf16'`.
* **Built on tt-metal**: pure C++ kernels, no extra Python bindings.